In [39]:
from google.colab import drive
drive.mount('/content/drive')

import os, cv2, numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

BASE = "/content/drive/MyDrive/coco20k"
TRAIN_IMG  = f"{BASE}/train/images"
TRAIN_MASK = f"{BASE}/train/masks"
VAL_IMG    = f"{BASE}/val/images"
VAL_MASK   = f"{BASE}/val/masks"

IMG_SIZE = 128
BATCH = 16
EPOCHS = 3

def get_pairs(img_dir, mask_dir):
    imgs = sorted(os.listdir(img_dir))
    paired = []
    for img in imgs:
        mask = img.replace('.jpg', '.png')
        if os.path.exists(os.path.join(mask_dir, mask)):
            paired.append(img)
    return paired

train_files = get_pairs(TRAIN_IMG, TRAIN_MASK)[:200]
val_files   = get_pairs(VAL_IMG, VAL_MASK)[:50]

def load_matched(img_dir, mask_dir, file_list):
    X, Y = [], []
    for f in file_list:
        img = cv2.imread(os.path.join(img_dir, f))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0
        mask = cv2.imread(os.path.join(mask_dir, f.replace('.jpg','.png')), 0)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
        mask = mask / 255.0
        X.append(img)
        Y.append(mask[..., None])
    return np.array(X), np.array(Y)

trainX, trainY = load_matched(TRAIN_IMG, TRAIN_MASK, train_files)
valX, valY     = load_matched(VAL_IMG, VAL_MASK, val_files)

def conv_block(x, f):
    x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
    return x

def build_unet():
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 3))
    c1 = conv_block(inp,16); p1 = layers.MaxPooling2D()(c1)
    c2 = conv_block(p1,32); p2 = layers.MaxPooling2D()(c2)
    c3 = conv_block(p2,64); p3 = layers.MaxPooling2D()(c3)
    bn = conv_block(p3,128)
    u1 = layers.UpSampling2D()(bn); u1 = layers.Concatenate()([u1, c3])
    c4 = conv_block(u1,64)
    u2 = layers.UpSampling2D()(c4); u2 = layers.Concatenate()([u2, c2])
    c5 = conv_block(u2,32)
    u3 = layers.UpSampling2D()(c5); u3 = layers.Concatenate()([u3, c1])
    c6 = conv_block(u3,16)
    out = layers.Conv2D(1,1,activation="sigmoid")(c6)
    return models.Model(inp, out)

model = build_unet()

def dice_coef(y_true, y_pred, smooth=1e-6):
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    inter = tf.reduce_sum(y_true_f * y_pred_f)
    return (2*inter + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def iou_metric(y_true, y_pred, smooth=1e-6):
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    inter = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - inter
    return (inter + smooth) / (union + smooth)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", dice_coef, iou_metric]
)

history = model.fit(
    trainX, trainY,
    validation_data=(valX, valY),
    epochs=EPOCHS,
    batch_size=BATCH
)

model.save("/content/drive/MyDrive/unet_task4_final.h5")
print("MODEL SAVED")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Epoch 1/3
13/13 ━━━━━━━━━━━━━━━━━━━━ 14s 551ms/step - accuracy: 0.5597 - dice_coef: 0.2321 - iou_metric: 0.1492 - loss: 0.6706 - val_accuracy: 0.6649 - val_dice_coef: 0.0055 - val_iou_metric: 0.0028 - val_loss: 0.6515
Epoch 2/3
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - accuracy: 0.6816 - dice_coef: 0.0044 - iou_metric: 0.0022 - loss: 0.6339 - val_accuracy: 0.6701 - val_dice_coef: 7.0302e-04 - val_iou_metric: 3.5173e-04 - val_loss: 0.6263
Epoch 3/3
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.7063 - dice_coef: 9.7470e-04 - iou_metric: 4.8763e-04 - loss: 0.5971 - val_accuracy: 0.6701 - val_dice_coef: 3.5802e-04 - val_iou_metric: 1.7906e-04 - val_loss: 0.6141


MODEL SAVED
